## AI4Climate ML tutorial - Inference and Visualisation
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-03-16
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered
In the previous notebooks we have explored and prepared a dataset for training a machine learning model, then we have trained a few different algorithms using this data. The next step is to use the trained model,  both to evaluate and understand how well it has learned the relationship in the data we want it to learn, but also then aplying to the intended use of the data. For example if we have trained a global climate model, we want to use the trained model for experiments around climate change and climate variability, for example.  In this notebook we will look at runnjing inference with the model and visualising the results.


### Prerequisites 
- Same as previous notebooks
- Have completed model training pipeline notebook


### Learning outcomes from completing the notebook
* Load a saved model
* Make predictions with the model
* Visualise the results

## Tutorial 
a balance of explanation and activity



In [1]:
import pathlib
import os
import datetime
import json

In [2]:
import pandas

In [3]:
import iris
import cartopy.crs

In [4]:
import matplotlib.pyplot

In [5]:
import mlflow

In [6]:
import sklearn
import sklearn.preprocessing
import sklearn.tree

## Exercises
for students to try that do not have solutions but maybe have an answer or benchmark to facilitate understanding


In [7]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'mo_linux',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/nopw/j04/mohc_shared/dscop/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, v

In [8]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path

In [9]:
current_platform = tutorial_config['platform']

In [10]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones')

In [11]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones/ml_ready')

In [12]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [13]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

### Load data for inference

In [14]:
current_res = 1.0

In [15]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones/ml_ready/climate_zones_1p0.csv')

In [16]:
zones_df = pandas.read_csv(mlready_data_path)

In [17]:
# reducing the total data point to decrease memeory requirements
# zones_df = zones_df[zones_df['scenario'] == 'historic']
zones_df = zones_df[(zones_df['period_start']==1991)&(zones_df['scenario']=='historic')]

In [18]:
zones_df

,lat,lon,precipitation_1.0_mean,precipitation_2.0_mean,precipitation_3.0_mean,precipitation_4.0_mean,precipitation_5.0_mean,precipitation_6.0_mean,precipitation_7.0_mean,precipitation_8.0_mean,...,air_temperature_10.0_std,air_temperature_11.0_std,air_temperature_12.0_std,kg_class,kg_confidence,period_start,period_end,scenario,climate_group,climate_subgroup
6514263,83.65,-35.55,20.3125,13.8750,14.8125,26.1875,15.6875,24.4375,30.5625,38.6875,...,5.8750,7.5625,8.4375,29.0,83.0,1991,2020,historic,E,ET
6514264,83.65,-34.55,22.3750,15.8125,17.4375,28.8750,19.3750,30.0625,36.8125,46.8750,...,7.3125,9.5625,10.5625,29.0,78.0,1991,2020,historic,E,ET
6514265,83.65,-34.45,19.6875,13.9375,15.3750,24.8750,17.2500,26.6250,32.5000,42.0000,...,7.6250,9.8750,10.9375,29.0,79.0,1991,2020,historic,E,ET
6514266,83.65,-34.35,17.8125,12.6250,13.8750,21.9375,15.1250,23.0000,28.3125,36.9375,...,7.6875,10.0625,11.1250,29.0,91.0,1991,2020,historic,E,ET
6514267,83.65,-34.25,17.3750,12.3125,13.5000,21.3125,14.5625,22.0000,27.1250,35.6250,...,8.9375,11.5625,12.8125,29.0,100.0,1991,2020,historic,E,ET
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8685679,-89.95,179.55,2.7500,3.3125,4.7500,5.0625,5.4375,4.1875,4.1875,6.6875,...,7.5625,6.3750,5.1875,30.0,94.0,1991,2020,historic,E,EF
8685680,-89.95,179.65,2.7500,3.3125,4.7500,5.0625,5.4375,4.1875,4.1875,6.6875,...,7.5625,6.3750,5.1875,30.0,94.0,1991,2020,historic,E,EF
8685681,-89.95,179.75,2.7500,3.3125,4.7500,5.0625,5.4375,4.1875,4.2500,6.6875,...,7.5625,6.3750,5.1875,30.0,94.0,1991,2020,historic,E,EF
8685682,-89.95,179.85,2.6875,3.3125,4.7500,5.0000,5.3750,4.1875,4.1875,6.6250,...,7.5625,6.3750,5.1875,30.0,94.0,1991,2020,historic,E,EF


In [19]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [20]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

['precipitation_1.0_mean',
 'precipitation_2.0_mean',
 'precipitation_3.0_mean',
 'precipitation_4.0_mean',
 'precipitation_5.0_mean',
 'precipitation_6.0_mean',
 'precipitation_7.0_mean',
 'precipitation_8.0_mean',
 'precipitation_9.0_mean',
 'precipitation_10.0_mean',
 'precipitation_11.0_mean',
 'precipitation_12.0_mean',
 'air_temperature_1.0_mean',
 'air_temperature_2.0_mean',
 'air_temperature_3.0_mean',
 'air_temperature_4.0_mean',
 'air_temperature_5.0_mean',
 'air_temperature_6.0_mean',
 'air_temperature_7.0_mean',
 'air_temperature_8.0_mean',
 'air_temperature_9.0_mean',
 'air_temperature_10.0_mean',
 'air_temperature_11.0_mean',
 'air_temperature_12.0_mean']

We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

In [21]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


In [22]:
random_seed = tutorial_config['random_seed']

In [23]:
test_frac = 0.1
val_frac = 0.1
val_frac_sub = (val_frac / (1.0-test_frac) )

In [24]:
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
remain_df = zones_df.drop(test_df.index)

In [25]:
val_df = remain_df.groupby(['period_start','scenario']).sample(frac=val_frac_sub, random_state=random_seed)
train_df = remain_df.drop(val_df.index)


In [26]:
input_scaler = sklearn.preprocessing.StandardScaler()
input_scaler.fit(train_df[predictors])


,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [27]:
input_scaler.mean_

array([43.42916013, 40.66081263, 45.37039227, 43.4316237 , 46.96939476,
       51.5951873 , 59.82245104, 59.87921732, 50.99005411, 46.77701885,
       42.34460638, 42.98790413, -6.73101179, -8.00810594, -7.94398391,
       -5.74286732, -2.88409237, -0.2428253 ,  0.51110559, -0.2300109 ,
       -2.10442286, -3.88600518, -5.09029401, -5.75483961])

In [28]:
X_train = input_scaler.transform(train_df[predictors])
X_val = input_scaler.transform(val_df[predictors])
X_test = input_scaler.transform(test_df[predictors])

In [55]:
train_df[[target_var]].value_counts()

climate_group
E                640014
D                433809
B                323017
A                198745
C                141552
Name: count, dtype: int64

In [54]:
target_encoder = sklearn.preprocessing.LabelEncoder()
target_encoder.fit(train_df[[target_var]])

/data/users/stephen.haddad/conda/ai4c_hack_spice/lib/python3.14/site-packages/sklearn/preprocessing/_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LabelEncoder()

In [56]:
y_train = target_encoder.transform(train_df[[target_var]])
y_val = target_encoder.transform(val_df[[target_var]])
y_test = target_encoder.transform(test_df[[target_var]])


/data/users/stephen.haddad/conda/ai4c_hack_spice/lib/python3.14/site-packages/sklearn/preprocessing/_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/data/users/stephen.haddad/conda/ai4c_hack_spice/lib/python3.14/site-packages/sklearn/preprocessing/_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/data/users/stephen.haddad/conda/ai4c_hack_spice/lib/python3.14/site-packages/sklearn/preprocessing/_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


In [57]:
dt_opts = {'max_depth':10, 'min_samples_leaf': 2, 'min_samples_split': 5}

In [58]:
dt_clf = sklearn.tree.DecisionTreeClassifier(**dt_opts)

In [59]:
%%time
dt_clf.fit(X_train, y_train) 

CPU times: user 27.2 s, sys: 78.9 ms, total: 27.3 s
Wall time: 27.6 s


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

In [60]:
dt_clf

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

In [61]:
y_pred_train = dt_clf.predict(X_train)
y_pred_val = dt_clf.predict(X_val)

### Next steps or potential follow on material



In [ ]:
v

###  Exmaples of Use


### Data statement
###     References
